# Can AI improve exoplanet candidate screening?

ASTR 12600 | Research notebook. AI assists with writing code; established BLS is not a new ML model. The random forest predicts NASA's *historical Robovetter labels*, not confirmed planets.

Run **Runtime > Run all** in Google Colab. NASA access is required for the last experiment; synthetic experiments run without the NASA catalog.


In [ ]:
%pip -q install numpy pandas scikit-learn astropy matplotlib requests
from pathlib import Path
Path('results').mkdir(exist_ok=True)


## 1. Reproducing a published transit-search method

These examples use simulated flux. A strong periodicity peak alone is not evidence of an actual planet: inspect the noise-only control.


In [ ]:
%%writefile exoplanet_detector.py
"""Reproducible transit-search demonstration (not a discovery claim).

AI authored; BLS is the published method of Kovacs, Zucker and Mazeh (2002).
Scientific calculations are implemented through Astropy, not a new ML model.
"""
from pathlib import Path
import argparse, json
import numpy as np
import matplotlib.pyplot as plt

def simulate_light_curve(seed=12600, planet=True, period_days=3.25,
                         depth=0.004, duration_days=0.16, noise=0.001):
    rng = np.random.default_rng(seed)
    t = np.arange(0, 65, 1/48)
    flux = 1 + rng.normal(0,noise,len(t))
    if planet:
        phase = (t - 0.92 + period_days/2)%period_days-period_days/2
        flux[np.abs(phase)<duration_days/2] -= depth
    return t, flux

def detect(t,flux,min_period=2.8,max_period=3.8):
    """Return strongest *candidate*, which can occur even without a planet."""
    from astropy.timeseries import BoxLeastSquares
    periods = np.linspace(min_period,max_period,2000)
    durations = np.array([0.08,0.12,0.16,0.20])
    model = BoxLeastSquares(t,flux,dy=np.full(len(t),0.001))
    power = model.power(periods,durations,objective="snr")
    i = int(np.argmax(power.power))
    result = {"best_period_days":float(power.period[i]),
              "depth_ppm":float(power.depth[i])*1e6,
              "transit_duration_hours":float(power.duration[i])*24,
              "peak_snr_statistic":float(power.power[i])}
    return result,periods,np.asarray(power.power)

def run(planet=True,seed=12600,out="results/transits",depth=0.004):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    t,flux=simulate_light_curve(seed=seed,planet=planet,depth=depth)
    result,periods,scores=detect(t,flux)
    result.update({"source":"SIMULATED, not real Kepler observations",
                   "injected_period_days":3.25 if planet else None,
                   "period_error_days":abs(result["best_period_days"]-3.25) if planet else None})
    label="planet_injected" if planet else "noise_only"
    fig,axes=plt.subplots(2,1,figsize=(8,6))
    axes[0].scatter(t,flux,s=1.5)
    axes[0].set(xlabel="Time (days)",ylabel="Relative stellar flux",
                title="Simulated stellar brightness")
    axes[1].plot(periods,scores,lw=1)
    axes[1].axvline(result["best_period_days"],color="gray",linestyle="--")
    axes[1].set(xlabel="Trial period (days)",ylabel="BLS detection score",
                title="Box Least Squares periodicity search")
    fig.tight_layout();fig.savefig(out/(label+".png"),dpi=170);plt.close(fig)
    (out/(label+".json")).write_text(json.dumps(result,indent=2))
    return result

def sensitivity_study(out="results/sensitivity",trials=12):
    """Keep noise level fixed and compare recovery at three transit depths."""
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    rows=[]
    for depth_ppm in (4000,1500,500):
        recovered=0
        errors=[]
        for i in range(trials):
            t,y=simulate_light_curve(seed=12600+i,depth=depth_ppm/1e6)
            result,_,_=detect(t,y)
            error=abs(result["best_period_days"]-3.25)
            errors.append(error)
            recovered+=error<0.05
        rows.append({"depth_ppm":depth_ppm,"recovered":recovered,
                     "trials":trials,"median_period_error_days":float(np.median(errors))})
    import pandas as pd
    df=pd.DataFrame(rows)
    df.to_csv(out/"sensitivity_summary.csv",index=False)
    fig,ax=plt.subplots(figsize=(6,3.5))
    ax.bar(df.depth_ppm.astype(str),100*df.recovered/df.trials)
    ax.set(xlabel="Simulated transit depth (ppm)",ylabel="Correct period recovered (%)",
           ylim=(0,105),title="Sensitivity to weaker transits")
    fig.tight_layout();fig.savefig(out/"sensitivity.png",dpi=170);plt.close(fig)
    return df

if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--output",default="results/transits")
    a=p.parse_args()
    print("Injected planet:",run(out=a.output))
    print("Noise-only:",run(planet=False,out=a.output))
    print(sensitivity_study())


In [ ]:
import exoplanet_detector as detector
from IPython.display import display, Image
positive=detector.run(out='results/transits')
negative=detector.run(planet=False,out='results/transits')
print('Injected transit:',positive)
print('Noise-only control:',negative)
display(Image(filename='results/transits/planet_injected.png'))
display(Image(filename='results/transits/noise_only.png'))


## 2. Controlled experiment: sensitivity to weaker simulated transits

The noise is held fixed and the injected transit depth is changed. This is an educational injection-recovery experiment, not a false-alarm calibration.


In [ ]:
sensitivity=detector.sensitivity_study(out='results/sensitivity',trials=12)
display(sensitivity)
display(Image(filename='results/sensitivity/sensitivity.png'))


## 3. Machine-learning extension: reduce misleading candidate flags

NASA Kepler DR25 catalog. Four astronomical measurements, one random forest, and one SNR-only baseline. The same star cannot appear in both training and test. Thresholds are set on validation stars. **No results should be reported until the real data run succeeds.**


In [ ]:
%%writefile ml_screening.py
"""Reproducible screening comparison on the historical Kepler DR25 KOI catalog.

This script predicts NASA's past Robovetter labels, NOT independently confirmed planets.
Generated with AI assistance; review AI_USE_LOG.md before citing results.
"""
from pathlib import Path
from urllib.parse import urlencode
import argparse, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

FEATURES = ["koi_period", "koi_duration", "koi_depth", "koi_model_snr"]
COLS = ["kepid", "kepoi_name", "koi_pdisposition"] + FEATURES
TABLE = "q1_q17_dr25_koi"

def nasa_url():
    query = "select " + ",".join(COLS) + " from " + TABLE
    return "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?" + urlencode({"query": query, "format": "csv"})

def load_data(path, download=False):
    path = Path(path)
    if not path.exists():
        if not download:
            raise FileNotFoundError("Download NASA CSV and save to " + str(path) + "\nURL: " + nasa_url())
        r = requests.get(nasa_url(), timeout=80)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    df = pd.read_csv(path, comment="#")
    missing = set(COLS) - set(df.columns)
    if missing:
        raise ValueError("Missing columns: " + str(missing))
    df = df.loc[df.koi_pdisposition.isin(["CANDIDATE", "FALSE POSITIVE"]), COLS].copy()
    df = df.dropna(subset=["kepid", "kepoi_name"]).drop_duplicates("kepoi_name")
    df["label"] = (df.koi_pdisposition == "CANDIDATE").astype(int)
    for col in FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[~np.isfinite(df[col]) | (df[col] < 0), col] = np.nan
    if len(df) < 500 or df.label.nunique() != 2:
        raise ValueError("Unexpected NASA catalog or insufficient labeled observations")
    return df.reset_index(drop=True)

def split_by_star(df, seed=12600):
    y, groups = df.label.to_numpy(), df.kepid.to_numpy()
    allidx = np.arange(len(df))
    split1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval, test = next(split1.split(allidx, y, groups))
    split2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed + 1)
    a, b = next(split2.split(trainval, y[trainval], groups[trainval]))
    train, valid = trainval[a], trainval[b]
    for left, right in [(train, valid), (train, test), (valid, test)]:
        assert not set(groups[left]).intersection(groups[right]), "Same-star data leakage!"
    for ix in (train, valid, test):
        assert len(np.unique(y[ix])) == 2
    return train, valid, test

def cutoff(scores, labels, desired=0.90):
    positives = np.asarray(scores)[np.asarray(labels) == 1]
    if not len(positives):
        raise ValueError("Missing positive validation cases")
    k = int(np.ceil(desired * len(positives))) - 1
    return float(np.sort(positives)[::-1][k])

def metrics(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=float(accuracy_score(y, pred)),
                recall=float(recall_score(y, pred, zero_division=0)),
                precision=float(precision_score(y, pred, zero_division=0)),
                false_positive_rate=float(fp / (fp + tn)),
                fraction_flagged=float(np.mean(pred)),
                true_positive=int(tp), false_positive=int(fp),
                true_negative=int(tn), false_negative=int(fn))

def experiment(df, output="results/ml"):
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    train, valid, test = split_by_star(df)
    x = np.log1p(df[FEATURES].clip(lower=0).astype(float))
    y = df.label.to_numpy()
    forest = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=120, max_depth=9,
                               min_samples_leaf=3, random_state=12600, n_jobs=-1))
    forest.fit(x.iloc[train], y[train])
    snr_median = float(np.nanmedian(x.iloc[train].koi_model_snr))
    def snr(ix):
        return x.iloc[ix].koi_model_snr.fillna(snr_median).to_numpy()
    alternatives = {
       "SNR only": (snr(valid), snr(test)),
       "Random forest": (forest.predict_proba(x.iloc[valid])[:, 1],
                         forest.predict_proba(x.iloc[test])[:, 1]),
    }
    report = {}; rows = []
    for name, (vscore, tscore) in alternatives.items():
        selected = cutoff(vscore, y[valid])
        vpred = vscore >= selected
        tpred = tscore >= selected
        report[name] = {"threshold": selected,
                        "validation": metrics(y[valid], vpred),
                        "test": metrics(y[test], tpred)}
        rows.append({"method": name, **report[name]["test"]})
        if name == "Random forest":
            cm = confusion_matrix(y[test], tpred, labels=[0,1])
            fig, ax = plt.subplots(figsize=(5,4))
            ax.imshow(cm, cmap="Blues")
            ax.set_xticks([0,1], ["Screen out","Review"])
            ax.set_yticks([0,1], ["NASA false positive","NASA candidate"])
            for (i,j), count in np.ndenumerate(cm):
                ax.text(j,i,str(count),ha="center",va="center")
            ax.set_title("Holdout-star confusion matrix")
            fig.tight_layout(); fig.savefig(out/"confusion_matrix.png",dpi=170); plt.close(fig)
    table = pd.DataFrame(rows)
    table.to_csv(out/"comparison.csv",index=False)
    fig, ax = plt.subplots(figsize=(7,3))
    ax.barh(table.method, 100*table.fraction_flagged)
    ax.set_xlim(0,100)
    ax.set_xlabel("Test signals flagged (%)")
    ax.set_title("SNR threshold versus random-forest screening")
    fig.tight_layout(); fig.savefig(out/"screening_comparison.png",dpi=170); plt.close(fig)
    payload = {"dataset": TABLE, "n_rows": len(df),
               "n_train": len(train), "n_validation": len(valid), "n_test": len(test),
               "label_caveat": "Historical Robovetter classification; not confirmed planets",
               "features": FEATURES, "results": report}
    (out/"metrics.json").write_text(json.dumps(payload,indent=2))
    return payload

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv",default="data/kepler_dr25.csv")
    parser.add_argument("--download",action="store_true")
    parser.add_argument("--output",default="results/ml")
    a = parser.parse_args()
    try:
        result = experiment(load_data(a.csv,a.download),a.output)
        print(json.dumps(result,indent=2))
    except Exception as exc:
        print("NASA test not completed:",str(exc))
        print("Manual CSV link:",nasa_url())
        raise


In [ ]:
from pathlib import Path
import ml_screening
csv=Path('data/kepler_dr25.csv')
print('Official NASA CSV URL: ',ml_screening.nasa_url())
try:
    df=ml_screening.load_data(csv,download=True)
    results=ml_screening.experiment(df,'results/ml')
    print('REAL NASA RESULTS:',results)
    display(Image(filename='results/ml/screening_comparison.png'))
    display(Image(filename='results/ml/confusion_matrix.png'))
except Exception as exc:
    print('NASA ANALYSIS NOT COMPLETED:',str(exc))
    print('If automatic download fails, open the URL above in your browser and upload its CSV:')
    try:
        from google.colab import files
        uploaded=files.upload()
        if uploaded:
            csv.parent.mkdir(parents=True,exist_ok=True)
            csv.write_bytes(next(iter(uploaded.values())))
            df=ml_screening.load_data(csv)
            results=ml_screening.experiment(df,'results/ml')
            print('REAL NASA RESULTS:',results)
    except Exception as err:
        print('Manual CSV fallback did not complete:',str(err))


### NASA catalog provenance

If the real NASA download succeeded, save the input CSV fingerprint (SHA-256), candidate/false-positive counts, number of distinct host stars, and missing-measurement summary. This makes the results auditable. The downloaded catalog is not a list of independently confirmed planets.


In [ ]:
%%writefile dataset_manifest.py
"""Create a scientific provenance manifest for a local NASA Kepler DR25 CSV.

This script does not train a model. It validates the required schema, records
the file hash, and reports label counts and missing measurements.
"""
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import argparse, hashlib, json
import numpy as np
import pandas as pd

SOURCE_TABLE="q1_q17_dr25_koi"
REQUIRED=("kepid","kepoi_name","koi_pdisposition",
          "koi_period","koi_duration","koi_depth","koi_model_snr")
FEATURES=REQUIRED[3:]
DISPOSITIONS=("CANDIDATE","FALSE POSITIVE")


def make_manifest(csv_path):
    path=Path(csv_path)
    if not path.is_file():
        raise FileNotFoundError(path)
    hasher=hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b""):
            hasher.update(chunk)
    data=pd.read_csv(path,comment="#")
    missing=set(REQUIRED)-set(data.columns)
    if missing:
        raise ValueError("Missing expected NASA columns: "+repr(sorted(missing)))
    retained=data.loc[data["koi_pdisposition"].isin(DISPOSITIONS)].copy()
    retained=retained.dropna(subset=["kepid","kepoi_name"]).drop_duplicates("kepoi_name")
    if len(retained)<500 or retained.koi_pdisposition.nunique()!=2:
        raise ValueError("Unexpected NASA data: incomplete or missing labels")
    numeric={}
    for col in FEATURES:
        values=pd.to_numeric(retained[col],errors="coerce")
        values=values.where(np.isfinite(values)&(values>=0))
        numeric[col]={"missing_count":int(values.isna().sum()),
                      "missing_fraction":float(values.isna().mean()),
                      "median":float(values.median()) if values.notna().any() else None}
    return {
        "source":"NASA Exoplanet Archive Kepler DR25 KOI catalog",
        "table":SOURCE_TABLE,
        "query_columns":list(REQUIRED),
        "generated_utc":datetime.now(timezone.utc).isoformat(),
        "csv_file_name":path.name,
        "csv_size_bytes":path.stat().st_size,
        "csv_sha256":hasher.hexdigest(),
        "rows_in_file":int(len(data)),
        "rows_after_labeled_deduplication":int(len(retained)),
        "distinct_host_stars":int(retained.kepid.nunique()),
        "disposition_counts":{k:int((retained.koi_pdisposition==k).sum()) for k in DISPOSITIONS},
        "measurements":numeric,
        "label_warning":"koi_pdisposition is an historical automatic vetter classification, not independently confirmed planet truth",
        "selection_warning":"Kepler Objects of Interest (KOI) are a preselected subset of all transit-like threshold crossings",
        "metadata_warning":"Store exact NASA TAP query URL separately; no alternate data versions should be silently substituted",
    }


def main():
    p=argparse.ArgumentParser()
    p.add_argument("--csv",default="data/kepler_dr25.csv")
    p.add_argument("--out",default="results/ml/data_manifest.json")
    args=p.parse_args()
    manifest=make_manifest(args.csv)
    dst=Path(args.out)
    dst.parent.mkdir(parents=True,exist_ok=True)
    dst.write_text(json.dumps(manifest,indent=2)+"\n")
    print(json.dumps({k:manifest[k] for k in ("csv_sha256","rows_in_file",
       "rows_after_labeled_deduplication","distinct_host_stars",
       "disposition_counts")},indent=2))


if __name__=="__main__":
    main()


In [ ]:
from pathlib import Path
if Path('data/kepler_dr25.csv').is_file():
    from dataset_manifest import make_manifest
    import json
    manifest=make_manifest('data/kepler_dr25.csv')
    Path('results/ml').mkdir(parents=True,exist_ok=True)
    Path('results/ml/data_manifest.json').write_text(json.dumps(manifest,indent=2))
    print('NASA dataset SHA-256:', manifest['csv_sha256'])
    print('Labeled rows:',manifest['rows_after_labeled_deduplication'])
else:
    print('NASA catalog not present: provenance manifest deferred.')


## 4. Blind transit-search stress test (new)

**This is an additional controlled simulation, not NASA survey results.** Previously, the detector searched around a period it effectively knew. This benchmark draws periods randomly from 2.1–9.8 days and searches a prespecified 1.5–12-day range. Null-only simulations select the flagging threshold; independent signals and controls are then tested. The quick run uses only four examples per condition, so its rates are illustrative rather than precise.


In [ ]:
%%writefile blind_transit_benchmark.py
"""Blind transit injection/recovery benchmark.

Scientific status: synthetic exercise, not NASA Kepler survey completeness.
BLS is a published algorithm provided by Astropy, not newly invented ML.
"""
from __future__ import annotations
import argparse, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PERIOD_SEARCH = (1.5, 12.0)
DURATIONS_DAYS = (0.09, 0.15, 0.24)
FALSE_ALARM_TARGET = .10
NO_PLANET_KINDS = ("white_noise", "red_noise", "single_dip")
TEST_KINDS = NO_PLANET_KINDS + ("binary_like",)
PLANET_DEPTHS_PPM = (500, 1500, 4000)


def simulate(seed, kind="planet", depth_ppm=1500, baseline=48., cadence=.03,
             measurement_sigma=.001):
    """Draw random period, phase, duration and missing data; includes controls."""
    if kind not in ("planet", *TEST_KINDS):
        raise ValueError("Unknown kind: " + str(kind))
    rng = np.random.default_rng(seed)
    t = np.arange(0, baseline, cadence)
    f = 1 + rng.normal(0, measurement_sigma, len(t))
    if kind in ("planet", "red_noise", "single_dip", "binary_like"):
        eps = rng.normal(0, .00018, len(t))
        red = np.zeros(len(t))
        for j in range(1, len(t)):
            red[j] = .94*red[j-1] + eps[j]
        f += red
        f += .00025 * np.sin(2*np.pi*t/rng.uniform(6,13)+rng.uniform(0,2*np.pi))
    truth = {"kind":kind, "seed":int(seed), "injected_period_days":None,
             "injected_depth_ppm":0., "injected_duration_days":None}
    if kind == "planet":
        period = float(rng.uniform(2.1, 9.8))
        duration = float(rng.uniform(.10,.20))
        epoch = float(rng.uniform(0.,period))
        phase = (t-epoch+period/2)%period-period/2
        f[np.abs(phase)<duration/2] -= depth_ppm/1e6
        truth.update(injected_period_days=period,
                     injected_depth_ppm=float(depth_ppm),
                     injected_duration_days=duration)
    elif kind == "single_dip":
        center=float(rng.uniform(5,baseline-5))
        duration=float(rng.uniform(.1,.22))
        f[np.abs(t-center)<duration/2] -= rng.uniform(.003,.007)
    elif kind == "binary_like":
        period=float(rng.uniform(4,9))
        epoch=float(rng.uniform(0,period))
        duration=float(rng.uniform(.1,.18))
        primary=(t-epoch+period/2)%period-period/2
        secondary=(t-epoch-period/2+period/2)%period-period/2
        f[np.abs(primary)<duration/2] -= .005
        f[np.abs(secondary)<duration/2] -= .0025
    # Shared gaps across all classes; no truth values passed into detector.
    mask = rng.random(len(t)) > .04
    for _ in range(3):
        start=rng.uniform(0,baseline-1)
        mask &= ~((t>=start)&(t<=start+rng.uniform(.12,.75)))
    return t[mask], f[mask], truth


def robust_sigma(flux):
    d=np.diff(np.asarray(flux,dtype=float))
    med=np.median(d)
    return float(max(np.median(np.abs(d-med))/(0.6744897501960817*np.sqrt(2)),1e-6))


def bls_peak(times, flux, bounds=PERIOD_SEARCH, frequency_factor=5.):
    """Blind period search over a predeclared broad range using Astropy BLS."""
    from astropy.timeseries import BoxLeastSquares
    times=np.asarray(times,dtype=float);flux=np.asarray(flux,dtype=float)
    if len(times)<100 or not np.all(np.isfinite(times+flux)):
        raise ValueError("Insufficient finite measurements")
    sigma=robust_sigma(flux)
    m=BoxLeastSquares(times,flux,dy=np.full_like(flux,sigma))
    r=m.autopower(DURATIONS_DAYS, minimum_period=bounds[0],
                  maximum_period=bounds[1], objective="snr",
                  frequency_factor=frequency_factor)
    k=int(np.nanargmax(r.power))
    return {"period_days":float(r.period[k]),"score":float(r.power[k]),
            "depth_ppm":float(r.depth[k])*1e6,
            "duration_days":float(r.duration[k]),"sigma_estimate":sigma}


def threshold_from_controls(scores, target_fpr=FALSE_ALARM_TARGET):
    """Only calibration controls set the threshold; not a survey-wide FAP."""
    x=np.asarray(scores,dtype=float)
    if len(x)<8 or not np.all(np.isfinite(x)):
        raise ValueError("Need >=8 finite calibration examples")
    if not 0<target_fpr<1:
        raise ValueError("target_fpr should be between 0 and 1")
    return float(np.quantile(x,1-target_fpr,method="higher"))


def summarize_trials(table):
    rows=[]
    for name,g in table.groupby("group",sort=False):
        n=len(g); selected=int(g.flagged.sum())
        correct=int(g.recovered.sum()) if name.startswith("planet_") else None
        rows.append({"group":name,"count":n,"flagged":selected,
                     "fraction_flagged":float(selected/n),
                     "correct_and_flagged":correct,
                     "recovery_fraction":float(correct/n) if correct is not None else None})
    return pd.DataFrame(rows)


def run_experiment(outdir="results/blind",seed=12600,per_condition=8,
                   target_fpr=FALSE_ALARM_TARGET,frequency_factor=5.):
    """Precalibrate null peak scores and evaluate independent holdout stars."""
    if per_condition<4:
        raise ValueError("Minimum 4 simulated stars per category")
    out=Path(outdir);out.mkdir(parents=True,exist_ok=True)
    n=per_condition*(len(NO_PLANET_KINDS)+len(TEST_KINDS)+len(PLANET_DEPTHS_PPM))
    samples=np.random.SeedSequence(seed).spawn(n)
    seeds=iter([int(s.generate_state(1)[0]) for s in samples])
    null_scores=[]
    for kind in NO_PLANET_KINDS:
        for _ in range(per_condition):
            t,f,_=simulate(next(seeds),kind=kind)
            null_scores.append(bls_peak(t,f,frequency_factor=frequency_factor)["score"])
    limit=threshold_from_controls(null_scores,target_fpr)
    records=[]
    conditions=[(kind,None) for kind in TEST_KINDS]
    conditions += [("planet",depth) for depth in PLANET_DEPTHS_PPM]
    for kind,depth in conditions:
        for _ in range(per_condition):
            s=next(seeds)
            t,f,truth=simulate(s,kind=kind,depth_ppm=depth or 1500)
            peak=bls_peak(t,f,frequency_factor=frequency_factor)
            flagged=bool(peak["score"]>limit)
            correct=bool(kind=="planet" and flagged and
                abs(peak["period_days"]-truth["injected_period_days"]) /
                truth["injected_period_days"] < .02)
            records.append({"seed":s,"group":f"planet_{depth}ppm" if kind=="planet" else kind,
                            "known_true_period_days":truth["injected_period_days"],
                            **peak,"flagged":flagged,"recovered":correct})
    trials=pd.DataFrame(records)
    trials.to_csv(out/"holdout_trials.csv",index=False)
    summary=summarize_trials(trials)
    summary.to_csv(out/"summary.csv",index=False)
    fig,ax=plt.subplots(figsize=(8,4.5))
    ax.bar(summary.group,summary.fraction_flagged)
    ax.set_ylim(0,1.05)
    ax.set_ylabel("Fraction of independent simulations")
    ax.set_title("Synthetic BLS: null-calibrated threshold, blind period")
    ax.tick_params(axis="x",rotation=25)
    fig.tight_layout()
    fig.savefig(out/"blind_detection_by_group.png",dpi=170);plt.close(fig)
    report={"source":"SYNTHETIC ONLY (not real Kepler results)",
            "search_period_days":list(PERIOD_SEARCH),
            "target_control_fpr":target_fpr,
            "empirical_null_threshold":limit,
            "calibration_count":len(null_scores),
            "holdout_count":len(trials),
            "limitations":"Simple noise surrogates; no real Kepler survey completeness or FAP",
            "summary":json.loads(summary.to_json(orient="records"))}
    (out/"method_and_results.json").write_text(json.dumps(report,indent=2))
    return report


if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--output",default="results/blind")
    p.add_argument("--per-condition",type=int,default=8)
    p.add_argument("--seed",type=int,default=12600)
    p.add_argument("--target-fpr",type=float,default=.10)
    args=p.parse_args()
    print(json.dumps(run_experiment(args.output,args.seed,args.per_condition,
                                    args.target_fpr),indent=2))


In [ ]:
import blind_transit_benchmark as blind
from IPython.display import display, Image
print('Running blind synthetic benchmark (no real NASA data)...')
blind_report=blind.run_experiment(outdir='results/blind',per_condition=4)
print(blind_report)
display(Image(filename='results/blind/blind_detection_by_group.png'))


### 4b. Inspect individual candidate transits (new)

A folded transit may be dominated by one bad observation. This heuristic diagnostic describes each candidate event and reports a **warning**, not a planet probability. It reuses the independently recovered BLS period and phase, not the hidden injection truth. Odd/even depth differences and event dominance are exploratory until tested against realistic observation artifacts.


In [ ]:
%%writefile transit_consistency.py
"""Exploratory per-event consistency checks for BLS transit candidates.

Inspired by established transit vetting (e.g., Robnik et al., MNRAS 2026).
These heuristic flags do not establish a planet or provide a calibrated FAP.
"""
import numpy as np
import pandas as pd


def individual_transit_checks(time,flux,period,epoch,duration,sigma=None):
    """Characterize successive dimming events at an independently fitted period.

    The local out-of-transit comparison uses 1.5–4 event durations.
    Missing or insufficiently sampled events are omitted, not marked as passes.
    """
    t=np.asarray(time,dtype=float);y=np.asarray(flux,dtype=float)
    if len(t)!=len(y) or not np.all(np.isfinite(t+y)):
        raise ValueError("Time and brightness arrays must have matching finite values")
    if not(np.isfinite(period) and np.isfinite(epoch) and np.isfinite(duration)):
        raise ValueError("Nonfinite ephemeris")
    if period<=0 or duration<=0 or duration>=period/3:
        raise ValueError("Implausible ephemeris or duration")
    if len(t)<20:
        raise ValueError("Too few points")
    if sigma is None:
        from blind_transit_benchmark import robust_sigma
        sigma=robust_sigma(y)
    sigma=float(sigma)
    if sigma<=0:
        raise ValueError("Invalid scatter estimate")
    events=[]
    kmin=int(np.floor((t.min()-epoch)/period))-1
    kmax=int(np.ceil((t.max()-epoch)/period))+1
    for number in range(kmin,kmax+1):
        center=float(epoch+number*period)
        dt=np.abs(t-center)
        mask_in=dt<duration/2
        mask_out=(dt>1.5*duration)&(dt<4*duration)
        nin=int(mask_in.sum());nout=int(mask_out.sum())
        if nin<3 or nout<6:
            continue
        depth=float(np.median(y[mask_out])-np.median(y[mask_in]))
        error=sigma*np.sqrt(1/nin+1/nout)
        events.append({"event":number,"center_day":center,"depth_ppm":depth*1e6,
                       "snr":depth/error,"n_in":nin,"n_out":nout})
    table=pd.DataFrame(events,columns=[
        "event","center_day","depth_ppm","snr","n_in","n_out"])
    if table.empty:
        return {"n_observable_events":0,"positive_depth_fraction":None,
                "dominance":None,"odd_even_depth_difference_ppm":None,
                "warning":"No adequately sampled individual events"},table
    positive_scores=np.maximum(table.snr.to_numpy(),0)**2
    dominance=(float(positive_scores.max()/positive_scores.sum())
               if positive_scores.sum()>0 else None)
    odd=table.loc[table.event%2==1,"depth_ppm"].to_numpy()
    even=table.loc[table.event%2==0,"depth_ppm"].to_numpy()
    odd_even=(float(np.median(odd)-np.median(even))
              if len(odd)>=2 and len(even)>=2 else None)
    fraction=float((table.depth_ppm>0).mean())
    warnings=[]
    if len(table)<3:
        warnings.append("fewer than three sampled transit events")
    if dominance is not None and dominance>.6:
        warnings.append("one event dominates positive-depth evidence")
    if fraction<.6:
        warnings.append("individual dips disagree with common transit template")
    return {"n_observable_events":int(len(table)),
            "positive_depth_fraction":fraction,
            "dominance":dominance,
            "odd_even_depth_difference_ppm":odd_even,
            "warning":"; ".join(warnings) if warnings else
            "No heuristic warning; this does NOT confirm a planet"},table


def inspect_from_bls(time,flux):
    """Fit period and epoch independently with blind BLS, then inspect events."""
    from blind_transit_benchmark import bls_peak
    peak=bls_peak(time,flux)
    result,table=individual_transit_checks(
        time,flux,peak["period_days"],peak["transit_time_days"],
        peak["duration_days"])
    return peak,result,table


In [ ]:
from transit_consistency import inspect_from_bls
for kind in ['planet','single_dip']:
    time, flux, truth=blind.simulate(seed=150,kind=kind,depth_ppm=4000)
    fitted,diagnostics,event_table=inspect_from_bls(time,flux)
    print(kind,'injected truth:',truth)
    print(kind,'recovered BLS:',fitted)
    print(kind,'event diagnostics:',diagnostics)
    display(event_table.head(12))


## 5. Uncertainty-aware ML triage (new)

This study reuses the historical DR25 catalog and compares two ways to decide which cases need independent human review. The underlying four random forests, predictions, and split by star remain identical. The only changed decision is which ambiguous signals get reviewed. Cutoffs are chosen using validation stars; held-out stars remain untouched. **Model agreement is a heuristic, not proof of a planet.**


In [ ]:
%%writefile ml_uncertainty_triage.py
"""Validation-selected review triage: margin-only versus model-disagreement.

Uses historical NASA Robovetter labels, not independently confirmed planets.
Classifier scores are NOT calibrated astronomical planet probabilities.
"""
import argparse, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import ml_screening as base

FEATURES=base.FEATURES


def select_review_cutoff(confidence,target_review=.30):
    values=np.asarray(confidence,dtype=float)
    if len(values)<10 or not np.isfinite(values).all():
        raise ValueError("Need >=10 valid validation confidence values")
    if not 0<target_review<1:
        raise ValueError("target review fraction must be in (0,1)")
    return float(np.quantile(values,target_review))


def triage_evaluation(y,pred,review):
    """All rows are counted; review is neither a success nor a planet detection."""
    y=np.asarray(y,dtype=int)
    pred=np.asarray(pred,dtype=bool)
    review=np.asarray(review,dtype=bool)
    if not(len(y)==len(pred)==len(review)):
        raise ValueError("Array lengths mismatch")
    auto=~review
    wrong=(pred!=y)
    return {"n":int(len(y)),"auto_n":int(auto.sum()),
            "review_n":int(review.sum()),
            "review_fraction":float(review.mean()),
            "auto_wrong_count":int(np.sum(wrong&auto)),
            "auto_error_rate":float(np.mean(wrong[auto])) if auto.any() else None,
            "false_dismissals_auto":int(np.sum((y==1)&(~pred)&auto)),
            "false_acceptances_auto":int(np.sum((y==0)&pred&auto)),
            "candidates_sent_to_review":int(np.sum((y==1)&review)),
            "false_positives_sent_to_review":int(np.sum((y==0)&review)),
            "underlying_binary":base.metrics(y,pred)}


def star_bootstrap_difference(stars,y,pred,baseline_review,enhanced_review,
                              seed=12600,reps=400):
    """Paired bootstrap by host star, preserving intra-star clustering."""
    stars=np.asarray(stars)
    unique,inverse=np.unique(stars,return_inverse=True)
    wrong=(np.asarray(pred)!=np.asarray(y))
    a=np.asarray(baseline_review,dtype=bool)
    b=np.asarray(enhanced_review,dtype=bool)
    rng=np.random.default_rng(seed)
    diffs=[]
    for _ in range(reps):
        drawn=rng.integers(0,len(unique),len(unique))
        weights=np.bincount(drawn,minlength=len(unique))[inverse]
        aa=(~a)*weights
        bb=(~b)*weights
        if aa.sum()==0 or bb.sum()==0:
            continue
        diffs.append(float(np.sum(wrong*bb)/bb.sum()-np.sum(wrong*aa)/aa.sum()))
    if len(diffs)<max(20,reps*.7):
        return {"ci95":None,"reason":"Not enough valid bootstrap draws"}
    return {"ci95":[float(z) for z in np.quantile(diffs,[.025,.975])],
            "direction":"enhanced minus margin-only: negative means less auto error",
            "bootstrap_draws":len(diffs)}




def fixed_budget_mask(uncertainty, fraction):
    """Rank examples by uncertainty WITHOUT labels and review exactly round(n*b)."""
    scores=np.asarray(uncertainty,dtype=float)
    if scores.ndim!=1 or not np.isfinite(scores).all():
        raise ValueError("Expected one finite uncertainty score per example")
    if not 0<=fraction<=1:
        raise ValueError("Budget outside [0,1]")
    n=int(np.floor(len(scores)*fraction+.5))
    selection=np.argsort(-scores,kind="stable")[:n]
    mask=np.zeros(len(scores),dtype=bool)
    mask[selection]=True
    return mask


def per_100_input_metrics(y, pred, review):
    """Review outcomes are unknown, NOT counted as correct classifications."""
    y=np.asarray(y,dtype=int)
    pred=np.asarray(pred,dtype=bool)
    review=np.asarray(review,dtype=bool)
    if not(len(y)==len(pred)==len(review)) or not len(y):
        raise ValueError("Expected equally sized nonempty arrays")
    auto=~review
    wrong=(pred!=y)&auto
    dismissed=(y==1)&(~pred)&auto
    false_accept=(y==0)&pred&auto
    return {
        "review_n":int(review.sum()),
        "review_fraction":float(review.mean()),
        "wrong_auto":int(wrong.sum()),
        "wrong_auto_per_100_input":float(100*wrong.mean()),
        "candidate_false_dismissals_auto":int(dismissed.sum()),
        "candidate_false_dismissals_per_100_candidates":(
            float(100*dismissed.sum()/(y==1).sum()) if np.any(y==1) else None),
        "false_positive_auto_per_100_input":float(100*false_accept.mean()),
        "auto_error_fraction":float(np.mean((pred!=y)[auto])) if auto.any() else None,
    }


def paired_budget_ci(stars,y,pred,a_review,b_review,seed=12600,reps=400):
    """Paired host-star bootstrap difference B minus A in errors per 100 inputs.

    The CI is conditional on fitted models and the realized held-out sample.
    """
    stars=np.asarray(stars)
    unique,inverse=np.unique(stars,return_inverse=True)
    y=np.asarray(y)
    wrong=(np.asarray(pred,dtype=bool)!=y)
    a=np.asarray(a_review,dtype=bool)
    b=np.asarray(b_review,dtype=bool)
    deltas=((wrong&(~b)).astype(float)-(wrong&(~a)).astype(float))
    rng=np.random.default_rng(seed)
    values=[]
    for _ in range(reps):
        ids=rng.integers(0,len(unique),len(unique))
        weights=np.bincount(ids,minlength=len(unique))[inverse]
        values.append(float(100*np.average(deltas,weights=weights)))
    return {"difference_B_minus_A_per_100":float(100*deltas.mean()),
            "ci95":[float(v) for v in np.quantile(values,[.025,.975])],
            "reps":reps,"resampling_unit":"Kepler host star"}


def fair_review_budget_comparison(y,pred,stars,margin,ensemble_stability,
                                  budgets=(.10,.20,.30,.40),seed=12600):
    """Identical binary predictions & exactly equal review counts per budget.

    Random-review baseline averages 100 label-blind random allocations.
    """
    y=np.asarray(y,dtype=int)
    pred=np.asarray(pred,dtype=bool)
    strategies={"margin_only":-np.asarray(margin,dtype=float),
                "margin_plus_disagreement":-np.asarray(ensemble_stability,dtype=float)}
    rows=[]
    reviews_at_30={}
    for budget in budgets:
        for name,uncertainty in strategies.items():
            mask=fixed_budget_mask(uncertainty,budget)
            metrics=per_100_input_metrics(y,pred,mask)
            rows.append({"budget":float(budget),"method":name,**metrics})
            if abs(budget-.30)<1e-8:
                reviews_at_30[name]=mask
        rng=np.random.default_rng(seed+int(budget*10000))
        random_rows=[]
        for _ in range(100):
            mask=fixed_budget_mask(rng.random(len(y)),budget)
            random_rows.append(per_100_input_metrics(y,pred,mask))
        # All methods review identical number of cases, even when budget*n rounds.
        average={}
        for key in ("wrong_auto","wrong_auto_per_100_input",
                    "candidate_false_dismissals_auto",
                    "candidate_false_dismissals_per_100_candidates",
                    "false_positive_auto_per_100_input","auto_error_fraction"):
            values=[item[key] for item in random_rows if item[key] is not None]
            average[key]=float(np.mean(values)) if values else None
        rows.append({"budget":float(budget),"method":"random_review",
                     "review_n":int(random_rows[0]["review_n"]),
                     "review_fraction":float(random_rows[0]["review_fraction"]),
                     **average,"random_replicates":100})
    comparison=pd.DataFrame(rows)
    paired=paired_budget_ci(stars,y,pred,reviews_at_30["margin_only"],
                  reviews_at_30["margin_plus_disagreement"],seed=seed) if (
                  "margin_only" in reviews_at_30) else None
    return comparison,paired,reviews_at_30


def experiment(df,output="results/ml_triage",seed=12600,review_target=.30):
    out=Path(output);out.mkdir(parents=True,exist_ok=True)
    train,valid,test=base.split_by_star(df,seed)
    y=df["label"].to_numpy(dtype=int)
    stars=df["kepid"].to_numpy()
    X=np.log1p(df[list(FEATURES)].clip(lower=0).astype(float))
    validation=[];holdout=[]
    for i in range(4):
        model=make_pipeline(SimpleImputer(strategy="median"),
            RandomForestClassifier(n_estimators=90,max_depth=9,
                min_samples_leaf=3,random_state=seed+i,n_jobs=-1))
        model.fit(X.iloc[train],y[train])
        validation.append(model.predict_proba(X.iloc[valid])[:,1])
        holdout.append(model.predict_proba(X.iloc[test])[:,1])
    v=np.asarray(validation)
    t=np.asarray(holdout)
    avg_val=v.mean(axis=0);avg_test=t.mean(axis=0)
    std_val=v.std(axis=0);std_test=t.std(axis=0)
    decision=base.cutoff(avg_val,y[valid],.90)
    margin_val=np.abs(avg_val-decision)
    margin_test=np.abs(avg_test-decision)
    # Select uncertainty weighting ONLY on validation stars.
    # Final test-star labels are never used to choose this parameter.
    val_decisions=avg_val>=decision
    weighting_grid=(.01,.02,.04,.08,.16)
    validation_trials=[]
    for weight in weighting_grid:
        uncertainty=-(margin_val/(weight+std_val))
        validation_review=fixed_budget_mask(uncertainty,.30)
        val_metrics=per_100_input_metrics(y[valid],val_decisions,validation_review)
        validation_trials.append({"weight":weight,
            "wrong_auto":val_metrics["wrong_auto"],
            "false_dismissals_auto":val_metrics["candidate_false_dismissals_auto"]})
    # Prefer the conventional margin rule on exact ties.
    chosen_weight=min(validation_trials,key=lambda x:(
        x["wrong_auto"],x["false_dismissals_auto"],-x["weight"]))["weight"]
    stability_val=margin_val/(chosen_weight+std_val)
    stability_test=margin_test/(chosen_weight+std_test)
    margin_cut=select_review_cutoff(margin_val,review_target)
    stability_cut=select_review_cutoff(stability_val,review_target)
    original_review=margin_test<margin_cut
    new_review=stability_test<stability_cut
    pred=avg_test>=decision
    a=triage_evaluation(y[test],pred,original_review)
    b=triage_evaluation(y[test],pred,new_review)
    report={"dataset":"q1_q17_dr25_koi",
            "label_caveat":"Robovetter judgments, not independently confirmed planets",
            "methods":{"margin_only":a,"margin_plus_disagreement":b},
            "train_n":len(train),"validation_n":len(valid),"test_n":len(test),
            "validation_recall_goal":.90,
            "validation_review_goal":review_target,
            "decision_threshold":float(decision),
            "validation_weight_trials":validation_trials,
            "chosen_disagreement_weight":chosen_weight,
            "paired_star_bootstrap":star_bootstrap_difference(
                stars[test],y[test],pred,original_review,new_review,seed),
            "caveat":"Not real-planet probabilities; no true NASA results until data loaded"}
    # Equal-budget comparison: holds binary predictions fixed across policies.
    # Disagreement stabilizer chosen using validation stars only.
    equal_budget, paired_budget, review_masks = fair_review_budget_comparison(
        y[test],pred,stars[test],margin_test,stability_test,seed=seed)
    equal_budget.to_csv(out/"equal_budget_comparison.csv",index=False)
    # Save every held-out score and review assignment for reproducibility.
    holdout=df.iloc[test][["kepid","kepoi_name","koi_period",
                           "koi_model_snr","label"]].copy()
    holdout["model_score"] = avg_test
    holdout["ensemble_disagreement"] = std_test
    holdout["predicted_candidate"] = pred.astype(int)
    holdout["review_margin_30pct"] = review_masks["margin_only"]
    holdout["review_disagreement_30pct"] = review_masks["margin_plus_disagreement"]
    holdout.to_csv(out/"holdout_predictions.csv",index=False)
    report["equal_budget"]={
        "budgets":[.10,.20,.30,.40],
        "strategies":["random_review","margin_only","margin_plus_disagreement"],
        "same_classification_predictions":True,
        "reviewed_cases_count_equal_by_budget":True,
        "paired_error_difference_at_30pct":paired_budget,
        "random_control_repetitions":100,
        "interpretation":"Reallocation of review, not improvement to underlying classifier"}
    fig2,ax2=plt.subplots(figsize=(7.2,4.2))
    for name,part in equal_budget.groupby("method"):
        ax2.plot(part.budget*100,part.wrong_auto_per_100_input,
                 marker="o",label=name.replace("_"," "))
    ax2.set(xlabel="Review budget (% of all signals)",
            ylabel="Incorrect automatic decisions per 100 original signals",
            title="Equal review budgets: same held-out stars")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(out/"equal_budget_tradeoff.png",dpi=170)
    plt.close(fig2)
    groups=[]
    for feature,edges,labels in [
        ("koi_period",[-np.inf,10,100,np.inf],["<10 d","10-100 d",">=100 d"]),
        ("koi_model_snr",[-np.inf,10,30,np.inf],["<10","10-30",">=30"])]:
        bins=pd.cut(df.iloc[test][feature],edges,labels=labels,right=False)
        for label in labels:
            mask=(bins==label).to_numpy(dtype=bool)
            if not mask.any():
                continue
            # Fair subgroup comparison at exactly 30% whole-sample review budget.
            old=triage_evaluation(y[test][mask],pred[mask],
                                  review_masks["margin_only"][mask])
            new=triage_evaluation(y[test][mask],pred[mask],
                                  review_masks["margin_plus_disagreement"][mask])
            groups.append({"variable":feature,"range":label,"n":int(mask.sum()),
                           "margin_review_fraction":old["review_fraction"],
                           "ensemble_review_fraction":new["review_fraction"],
                           "margin_auto_wrong":old["auto_wrong_count"],
                           "ensemble_auto_wrong":new["auto_wrong_count"],
                           "small_group_warning":bool(mask.sum()<25)})
    pd.DataFrame(groups).to_csv(out/"subgroups.csv",index=False)
    pd.DataFrame([
       {"method":name,**{k:val for k,val in d.items() if k!="underlying_binary"}}
       for name,d in report["methods"].items()
    ]).to_csv(out/"triage_summary.csv",index=False)
    fig,ax=plt.subplots(figsize=(7,3.6))
    err=[100*a["auto_error_rate"] if a["auto_error_rate"] is not None else np.nan,
         100*b["auto_error_rate"] if b["auto_error_rate"] is not None else np.nan]
    review=[100*a["review_fraction"],100*b["review_fraction"]]
    pos=np.arange(2)
    ax.bar(pos-.16,err,width=.32,label="Error among auto-decided (%)")
    ax.bar(pos+.16,review,width=.32,label="Sent for review (%)")
    ax.set_xticks(pos,["Margin","Margin + disagreement"])
    ax.set_ylabel("Percent")
    ax.set_title("Held-out star classifications: report the review tradeoff")
    ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(out/"triage_tradeoff.png",dpi=170);plt.close(fig)
    (out/"triage_metrics.json").write_text(json.dumps(report,indent=2))
    return report


if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--csv",default="data/kepler_dr25.csv")
    p.add_argument("--download",action="store_true")
    p.add_argument("--output",default="results/ml_triage")
    a=p.parse_args()
    print(json.dumps(experiment(base.load_data(a.csv,a.download),a.output),indent=2))


In [ ]:
from pathlib import Path
import ml_uncertainty_triage as triage
if Path('data/kepler_dr25.csv').exists():
    try:
        current=ml_screening.load_data('data/kepler_dr25.csv')
        triage_report=triage.experiment(current,'results/ml_triage')
        print('REAL DR25 label-screening experiment:',triage_report)
        display(Image(filename='results/ml_triage/triage_tradeoff.png'))
    except Exception as error:
        print('DR25 triage not completed:',type(error).__name__,str(error))
else:
    print('NASA catalog missing: triage remains untested on real data. Follow the CSV download instructions above.')

if Path('results/ml_triage/equal_budget_tradeoff.png').exists():
    display(Image(filename='results/ml_triage/equal_budget_tradeoff.png'))
    display(__import__('pandas').read_csv('results/ml_triage/equal_budget_comparison.csv'))
if Path('results/ml_triage/holdout_predictions.csv').exists():
    print('Held-out predictions saved for audit:', 'results/ml_triage/holdout_predictions.csv')


## 6. Download evidence and write your own interpretation

Clearly distinguish simulated results from NASA catalog results. Report held-out candidate recall, false-positive rate, and fraction flagged for review. Explain what AI wrote, and what you checked independently.


In [ ]:
import shutil
from IPython.display import FileLink
result_zip=shutil.make_archive('ASTR_exoplanet_results','zip','results')
display(FileLink(result_zip))
